# 3 · Collect Statistics
Reads EMPIAR/EMD metadata, applies manual corrections, computes per-experiment CryoSPARC parameters, and writes a summary Parquet file.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
PARTICLES_CSV = "./metadata/EMPIAR_search_results_particles.csv"
SFM_CSV       = "./metadata/EMPIAR_search_results_single_frame.csv"
JOBS_XLSX     = "./metadata/All_Jobs.xlsx"
OUT_PARQUET   = "./metadata/EMPIAR_statistics.parquet"
N_WORKERS     = 10
BOX_MULTIPLIER = 2   # multiplier for computing extraction box size

In [2]:
import csv
import numpy as np
import pandas as pd
from utils.utils import box_sizes
from utils.scraping_utils import fetch_empiar_emd_metadata_and_pdb_data_multiprocess

## EMPIAR ID lists

In [3]:
MICROGRAPH_ENTRIES = (
    '10004', '10005', '10012', '10017', '10025', '10056', '10061', '10075', '10122', '10160',
    '10175', '10189', '10190', '10192', '10193', '10199', '10203', '10208', '10217', '10240',
    '10268', '10271', '10283', '10289', '10290', '10291', '10379', '10401', '10411', '10425',
    '10433', '10467', '10475', '10489', '10519', '10560', '10590', '10652', '10667', '10706',
    '10707', '10735', '10760', '10790', '10794', '10800', '10882', '10888', '10890', '10891',
    '10893', '10919', '10920', '10930', '11060', '11131', '11139', '11146', '11149', '11167',
    '11193', '11194', '11204', '11267', '11268', '11341', '11374', '11443', '11501', '11524',
    '11553', '11556', '11567', '11604', '11605', '11607', '11608', '11615', '11703', '11719',
    '11726', '11755', '11761', '11763', '11768', '11789', '11795', '11840', '11847', '11954',
    '11986', '12087', '12106', '12140', '12143', '12171', '12237', '12240', '12310', '12311',
    '12486', '12561', '12562', '12667',
)
PARTICLE_ENTRIES = (
    '10024', '10028', '10044', '10049', '10059', '10063', '10072', '10076', '10081', '10090',
    '10091', '10093', '10096', '10097', '10099', '10107', '10123', '10124', '10127', '10128',
    '10166', '10176', '10180', '10202', '10229', '10254', '10264', '10278', '10279', '10285',
    '10294', '10299', '10307', '10308', '10309', '10317', '10328', '10330', '10333', '10335',
    '10336', '10341', '10342', '10344', '10345', '10347', '10350', '10357', '10358', '10373',
    '10374', '10380', '10391', '10395', '10396', '10397', '10398', '10399', '10406', '10407',
    '10409', '10420', '10421', '10437', '10443', '10454', '10455', '10465', '10470', '10481',
    '10482', '10483', '10532', '10536', '10640', '10659', '10669', '10697', '10703', '10722',
    '10739', '10752', '10770', '10786', '10792', '10810', '10841', '10873', '10874', '10876',
    '11000', '11005', '11043', '11120', '11128', '11211', '11233', '11247', '11270', '11283',
    '11362', '11521', '11522', '11523', '11526', '11618', '11665', '11681', '11706', '11720',
    '11734', '11762', '11791', '11792', '11796', '11797', '11834', '11836', '11844', '11898',
    '11910', '11925', '11987', '12036', '12093', '12094', '12097', '12121', '12180', '12443',
    '12444', '12510',
)
ALL_ENTRIES = set(MICROGRAPH_ENTRIES + PARTICLE_ENTRIES)
print(f"Total entries: {len(ALL_ENTRIES)}")

Total entries: 236


## Manual correction tables

In [4]:
CHANGE_REPORTED_BOX             = {'11847': 256, '11116': 224, '10199': 600}
CHANGE_REPORTED_MIN_MAX         = {
    '11898':[420,420], '11341':[50,140],  '11405':[240,340], '11443':[60,110],
    '11485':[80,100],  '11564':[100,115], '11567':[60,100],  '11556':[90,90],
    '12140':[90,170],  '10017':[140,175], '10025':[115,160], '10096':[70,120],
    '10097':[70,135],  '10122':[130,130], '10175':[75,130],  '10189':[1300,1300],
    '10198':[330,330], '10217':[100,140], '10271':[180,200], '10373':[70,170],
    '10379':[100,105], '10590':[170,200], '10704':[70,190],  '10800':[75,120],
    '10801':[105,130], '10888':[850,850], '10890':[750,750], '11024':[200,230],
    '11077':[300,340], '10241':[130,300], '12240':[120,120], '12237':[200,305],
    '11791':[260,270], '11792':[90,90],   '11796':[130,130], '11693':[125,420],
    '10876':[80,170],  '10268':[200,260], '10215':[120,160], '10075':[275,285],
    '10160':[290,320], '11034':[310,380], '11354':[120,165], '10652':[300,300],
    '10005':[100,115], '10193':[480,500], '11194':[55,95],   '11681':[70,115],
    '11834':[120,175], '11618':[100,130], '11120':[80,180],  '10841':[200,220],
    '10752':[120,215], '10128':[100,125], '10420':[80,115],  '10455':[850,850],
    '10335':[45,55],   '10294':[750,750], '10229':[55,85],   '10099':[120,140],
    '10076':[200,205],
}
CHANGE_SYMMETRY                 = {
    '11768':'D2', '11501':'C2', '10652':'I',  '10075':'I',  '10799':'C2',
    '11861':'C2', '12171':'C2', '10005':'C4', '10012':'D2', '10025':'D7',
    '10189':'I',  '10193':'I',  '10203':'I',  '10240':'C2', '10425':'C2',
    '10760':'C7', '10801':'C2', '10893':'C13','11149':'C2', '11719':'C2',
    '12240':'O',  '10790':'C6',
}
CHANGE_ACCELERATION_VOLTAGE_KV  = {'10652': 300}
CHANGE_NOMINAL_CS_MM            = {
    '11986':2.7, '12562':2.7, '11847':2.7, '12140':2.7, '11755':2.7,
    '11556':2.7, '11763':2.7, '12106':2.7, '12087':2.7, '11898':2.7,
    '11501':2.7, '10735':2.7, '10652':2.7, '10075':2.7, '10203':2.7, '10379':2.0,
}
CHANGE_MICROGRAPH_PIXEL_SIZE    = {'11443': 0.36}
CHANGE_ELECTRON_DOSE_PER_IMAGE  = {'11768': 22.5, '10652': 47, '10429': 92}
NO_PS_DICT = {
    '11986':0.88,  '11755':1.069, '12106':0.85,  '12087':1.094, '11726':1.083,
    '11341':0.826, '10707':0.85,  '10706':1.0277,'10704':1.35,  '10667':1.35,
    '10291':1.232, '10289':1.232, '10290':1.232, '10285':1.0,   '10241':1.096,
    '10240':1.096, '10239':1.096, '11906':0.74,  '11608':0.835, '10937':1.71,
    '11120':0.83,  '10483':1.07,  '10299':1.40,
}
KEYS_TO_EXTRACT = ['acceleration_voltage_kV', 'nominal_cs_mm', 'electron_dose_per_image']

## 1 · Read CSVs and fetch metadata

In [5]:
with open(PARTICLES_CSV, newline='', encoding='utf-8') as f:
    csv_particles = np.array(list(csv.reader(f)))
particle_ids = set(csv_particles[1:, 0])

with open(SFM_CSV, newline='', encoding='utf-8') as f:
    sfm_rows = [row for row in csv.reader(f) if row[0] not in particle_ids]
csv_sfm  = np.array(sfm_rows)
csv_info = np.concatenate((csv_particles, csv_sfm[1:]))

empiar_emd_dict = {}
for entry in csv_info[1:]:
    empiar = entry[0].split('-')[-1]
    if empiar not in ALL_ENTRIES:
        continue
    parts = entry[10].split(',')
    emd   = [i.split('-')[-1] for i in parts] if len(parts) > 1 else [parts[0].split('-')[-1]]
    empiar_emd_dict[empiar] = emd

ALL_CATEGORIES = (
    'micrographs - single frame',
    'picked particles - single frame - unprocessed',
    'picked particles - single frame - processed',
    'picked particles - multiframe - processed',
    'picked particles - multiframe - unprocessed',
)

info = fetch_empiar_emd_metadata_and_pdb_data_multiprocess(
    empiar_emd_dict, N_WORKERS,
    use_local=True, category_filter=ALL_CATEGORIES, download_pdb=False,
)
print(f"Fetched {len(info)} entries")

XML file not found locally at ./metadata/emdb_xml/emd-23645-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23649-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23648-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23650-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23644-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23647-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23643-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23642-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23646-v30.xml
XML file not found locally at ./metadata/emdb_xml/emd-23651-v30.xml
Fetched 236 entries


## 2 · Filter to selected EMPIAR/EMD pairs from Excel

In [6]:
df = pd.read_excel(JOBS_XLSX)
selected = np.array([
    [str(r[0]).split('_')[0], str(r[1])]
    for r in zip(df['EMPIAR_ID'], df['EMD_ID'])
    if str(r[0]).split('_')[0] != 'Kastritis'
])
selected[np.where(selected[:, 0] == '10407')[0], 1] = '10898'

info_dict = {}
for item in info:
    if not item:
        continue
    empiar_id = next(iter(item))
    info_dict[empiar_id] = item[empiar_id]
    match = np.where(selected[:, 0] == empiar_id)[0]
    if len(match):
        sel_emd = str(selected[match[0], 1])
        if sel_emd in info_dict[empiar_id]['emd_info']:
            info_dict[empiar_id]['emd_info'] = {sel_emd: info_dict[empiar_id]['emd_info'][sel_emd]}

print(f"info_dict size: {len(info_dict)}")

info_dict size: 236


## 3 · Compute CryoSPARC statistics

In [7]:
cryosparc_dict = {}
test_box_sizes = []

for i in info_dict:
    result = {}
    temp_emp_key = next(iter(info_dict[i]['empiar_info']))
    temp_emp     = info_dict[i]['empiar_info'][temp_emp_key]
    temp_emd     = dict(next(iter(info_dict[i]['emd_info'].values())))

    # Pixel size
    emd_ps    = np.array(list(temp_emd['pixel_spacing'].values()), dtype=float)
    all_equal = np.all(emd_ps == emd_ps[0])
    pw, ph    = temp_emp['pixelWidth'], temp_emp['pixelHeight']

    if pw == ph:
        try:
            result['micrographs_pixel_size'] = float(pw)
        except (TypeError, ValueError):
            result['micrographs_pixel_size'] = NO_PS_DICT[i]
    elif pw == 'variable':
        result['micrographs_pixel_size'] = float(ph)
    elif ph == 'variable':
        result['micrographs_pixel_size'] = float(pw)
    else:
        print(f"EMPIAR {i}: pixel width/height mismatch ({pw} vs {ph})")
        result['authors_map_pixel_size'] = float(emd_ps[0])

    if all_equal:
        result['authors_map_pixel_size'] = float(emd_ps[0])
    else:
        print(f"EMPIAR {i}: unequal pixel spacings in EMD map")

    if i in CHANGE_MICROGRAPH_PIXEL_SIZE:
        result['micrographs_pixel_size'] = CHANGE_MICROGRAPH_PIXEL_SIZE[i]

    # Beam / dose
    result.update({k: temp_emd[k] for k in KEYS_TO_EXTRACT if k in temp_emd})
    if i in CHANGE_ACCELERATION_VOLTAGE_KV:
        result['acceleration_voltage_kV'] = str(CHANGE_ACCELERATION_VOLTAGE_KV[i])
    if i in CHANGE_NOMINAL_CS_MM:
        result['nominal_cs_mm'] = str(CHANGE_NOMINAL_CS_MM[i])
    if i in CHANGE_ELECTRON_DOSE_PER_IMAGE:
        result['electron_dose_per_image'] = str(CHANGE_ELECTRON_DOSE_PER_IMAGE[i])

    # Particle diameter
    temp_min, temp_max = 1_000_000, 0
    for mm in info_dict[i]['pdb_info'].values():
        temp_min = min(temp_min, mm['min_diameter'])
        temp_max = max(temp_max, mm['max_diameter'])
    if i in ('11405', '10271'):
        temp_min, temp_max = 1_000_000, 0
    if temp_min == 1_000_000 and temp_max == 0:
        result['min_diameter'] = result['max_diameter'] = 0
    else:
        result['min_diameter'], result['max_diameter'] = int(temp_min), int(temp_max)
    if i in CHANGE_REPORTED_MIN_MAX:
        result['min_diameter'], result['max_diameter'] = CHANGE_REPORTED_MIN_MAX[i]

    # Box size
    emd_box = int(np.max(np.array(list(temp_emd['dimensions'].values()), dtype=int)))
    result['authors_extraction_box_size'] = emd_box
    if i in CHANGE_REPORTED_BOX:
        result['authors_extraction_box_size'] = CHANGE_REPORTED_BOX[i]

    min_box = 0
    try:
        if result['max_diameter']:
            raw     = BOX_MULTIPLIER * (result['max_diameter'] / float(result['micrographs_pixel_size']))
            min_box = int(box_sizes[np.where(box_sizes / raw > 1)[0][0]])
    except Exception:
        pass

    if not min_box:
        mic_ps  = result.get('micrographs_pixel_size', 0)
        auth_ps = result.get('authors_map_pixel_size', 0)
        if mic_ps and auth_ps and abs(mic_ps - auth_ps) > 0.1:
            raw = emd_box / (mic_ps / auth_ps)
            min_box = int(box_sizes[np.where(box_sizes / raw > 1)[0][0]]) if raw < 4096 else 'too large'

    result['computed_extraction_box_size'] = min_box
    test_box_sizes.append([i, result['authors_extraction_box_size'], min_box, emd_ps[0],
                           result.get('micrographs_pixel_size', 0)])

    # Symmetry
    result['symmetry'] = temp_emd.get('symmetry')
    if i in CHANGE_SYMMETRY:
        result['symmetry'] = CHANGE_SYMMETRY[i]

    cryosparc_dict[i] = result

test_box_sizes = np.round(np.array(test_box_sizes).astype(float), 2)

## 4 · Inspect and save

In [8]:
cryosparc_df = pd.DataFrame.from_dict(cryosparc_dict, orient='index')
cryosparc_df['EMPIAR_ID'] = cryosparc_df.index
cryosparc_df

,micrographs_pixel_size,authors_map_pixel_size,acceleration_voltage_kV,nominal_cs_mm,electron_dose_per_image,min_diameter,max_diameter,authors_extraction_box_size,computed_extraction_box_size,symmetry,EMPIAR_ID
12094,1.1600,0.9670,300,0.01,45.0,0,0,432,384,C1,12094
11987,0.8180,0.8200,200,2.7,39.88,0,0,288,0,C4,11987
12180,0.7650,0.7650,300,,60.0,0,0,320,0,,12180
11844,0.8200,0.8200,300,2.7,57.25,0,0,256,0,C1,11844
12444,0.8600,0.8600,300,2.7,40.0,0,0,512,0,,12444
...,...,...,...,...,...,...,...,...,...,...,...
10025,0.6600,0.9820,300,2.7,53,115,160,300,512,D7,10025
10017,1.7700,1.7700,300,2,24,140,175,180,208,D2,10017
10012,0.6375,0.6375,300,2.7,45,0,0,340,0,D2,10012
10005,1.2156,1.2156,300,2.0,21,100,115,256,192,C4,10005


In [9]:
cryosparc_df.to_parquet(OUT_PARQUET, index=False)
print(f"Saved {len(cryosparc_df)} rows → {OUT_PARQUET}")

Saved 236 rows → ./metadata/EMPIAR_statistics.parquet
